# Final Task B 17-class - ResNet-50

Train ResNet-50 directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
task-B-17-new: /workspace/woodbench/notebooks/phase 1/task-B-17-new


In [3]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [4]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "resnet50"
DISPLAY_NAME = "ResNet-50"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: ResNet-50 (resnet50)
Batch size: 64
Data root: /workspace/woodbench/data/task-B-group-split


In [5]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-new/final

main/seed_42/resnet50
[main | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8408, Acc: 4.72% | Val Loss: 2.8509, Acc: 3.77% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.77%)


Epoch 2/50 | Train Loss: 2.8313, Acc: 6.61% | Val Loss: 2.8309, Acc: 5.76% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.76%)


Epoch 3/50 | Train Loss: 2.8041, Acc: 12.23% | Val Loss: 2.7965, Acc: 8.42% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.42%)


Epoch 4/50 | Train Loss: 2.7631, Acc: 20.22% | Val Loss: 2.7465, Acc: 18.05% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 18.05%)


Epoch 5/50 | Train Loss: 2.6985, Acc: 29.81% | Val Loss: 2.6720, Acc: 30.79% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 30.79%)


... (142 outputs left out)

Epoch 48/50 | Train Loss: 0.8117, Acc: 94.11% | Val Loss: 0.8974, Acc: 89.48% | LR: 4.87e-07[EarlyStop 8/10]


Epoch 49/50 | Train Loss: 0.8146, Acc: 93.54% | Val Loss: 0.8970, Acc: 89.81% | LR: 1.22e-07[EarlyStop 9/10]


Epoch 50/50 | Train Loss: 0.8090, Acc: 94.05% | Val Loss: 0.8930, Acc: 89.70% | LR: 0.00e+00[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 50.
[ERROR] main/seed_456/resnet50: RuntimeError('[enforce fail at inline_container.cc:747] . open file failed with strerror: No space left on device')

taskB_90rotation/seed_42/resnet50
[ERROR] taskB_90rotation/seed_42/resnet50: OSError(28, 'No space left on device')

taskB_label_smoothing_0.0/seed_42/resnet50
[ERROR] taskB_label_smoothing_0.0/seed_42/resnet50: OSError(28, 'No space left on device')

taskB_label_smoothing_0.2/seed_42/resnet50
[ERROR] taskB_label_smoothing_0.2/seed_42/resnet50: OSError(28, 'No space left on device')


RuntimeError: resnet50 failed runs: {'main/seed_456/resnet50': "RuntimeError('[enforce fail at inline_container.cc:747] . open file failed with strerror: No space left on device')", 'taskB_90rotation/seed_42/resnet50': "OSError(28, 'No space left on device')", 'taskB_label_smoothing_0.0/seed_42/resnet50': "OSError(28, 'No space left on device')", 'taskB_label_smoothing_0.2/seed_42/resnet50': "OSError(28, 'No space left on device')"}

In [ ]:
summarize_results(results)


## Continue after disk-full interruption

Run the setup/data-check cells above, but do not rerun the old `run_all_experiments` cell. The main seed-456 training reached epoch 50 and only the final `last.pth` save failed, so finalize it from the valid best checkpoint. Use `run_remaining` for the three ablations that still need training. Run one model notebook at a time.

In [9]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [10]:
from final_b_17class import (
    EXPERIMENTS,
    experiment_output_dir,
    load_completed_results,
    train_one,
)

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

def finalize_from_best(experiment_name, seed):
    output_dir = experiment_output_dir(ROOT, experiment_name, seed, BACKBONE)
    best_path = output_dir / f"{BACKBONE}_seed{seed}_best.pth"
    if not best_path.exists():
        raise FileNotFoundError(f"Missing best checkpoint: {best_path}")
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=False,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 2/6 completed configs for resnet50.


In [11]:
# Training already reached epoch 50; only evaluate/register the valid best checkpoint.
finalize_from_best("main", 456)


[main | resnet50 s456] Found checkpoint: /workspace/woodbench/data/experiments/Task-B-new/final/main/seed_456/resnet50/resnet50_seed456_best.pth
[main | resnet50 s456] test_acc=91.73% F1=92.05%


{'backbone': 'resnet50',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'best_val_acc': 90.14396456256921,
 'best_epoch': 40,
 'final_train_acc': 94.35096153846153,
 'final_val_acc': 90.14396456256921,
 'stopped_early': False,
 'epochs_trained': 40,
 'test_accuracy': 0.9172932330827067,
 'test_f1_macro': 0.9205189474497715,
 'test_precision_macro': 0.9310428014982095,
 'test_recall_macro': 0.9210133831785244,
 'test_f1_per_class': {'Araucaria araucana': 0.9285714285714286,
  'Callitris columellaris': 0.9230769230769231,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 0.7764705882352941,
  'Cryptomeria japonica': 0.9846153846153847,
  'Cunninghamia lanceolata': 0.9811320754716981,
  'Fokienia hodginsii': 1.0,
  'Picea glauca': 1.0,
  'Pinus cernua': 0.9508196721311475,
  'Pinus dalatensis': 0.8985507246376812,
  'Pinus krempfii': 0.8695652173913043,
  'Pinus radiata': 0.9393939393939394,
  'Podocarpus pilgeri': 0.5901639344262295,
  'Pseudotsuga brevifolia': 0.983050

In [12]:
# Remaining 90-degree rotation ablation
run_remaining("taskB_90rotation", 42)


[taskB_90rotation | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8382, Acc: 5.44% | Val Loss: 2.8417, Acc: 4.10% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 4.10%)


Epoch 2/50 | Train Loss: 2.8260, Acc: 6.91% | Val Loss: 2.8126, Acc: 8.42% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.42%)


Epoch 3/50 | Train Loss: 2.7910, Acc: 14.87% | Val Loss: 2.7586, Acc: 22.15% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 22.15%)


Epoch 4/50 | Train Loss: 2.7346, Acc: 26.77% | Val Loss: 2.6791, Acc: 34.55% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 34.55%)


Epoch 5/50 | Train Loss: 2.6570, Acc: 38.37% | Val Loss: 2.5777, Acc: 47.95% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 47.95%)


... (32 outputs left out)

Epoch 38/50 | Train Loss: 0.8322, Acc: 93.66% | Val Loss: 0.8315, Acc: 91.92% | LR: 1.65e-05[EarlyStop 8/10]


Epoch 39/50 | Train Loss: 0.8287, Acc: 94.02% | Val Loss: 0.8317, Acc: 91.69% | LR: 1.40e-05[EarlyStop 9/10]


Epoch 40/50 | Train Loss: 0.8195, Acc: 94.20% | Val Loss: 0.8269, Acc: 91.92% | LR: 1.17e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 40.
[taskB_90rotation | resnet50 s42] val_acc=92.14% test_acc=95.86% F1=95.80%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 92.13732004429679,
 'best_epoch': 30,
 'final_train_acc': 94.20072115384616,
 'final_val_acc': 91.91583610188262,
 'stopped_early': True,
 'epochs_trained': 40,
 'test_accuracy': 0.9586466165413534,
 'test_f1_macro': 0.9579671933015231,
 'test_precision_macro': 0.9639158956801536,
 'test_recall_macro': 0.9571759358635684,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9850746268656716,
  'Calocedrus rupestris': 1.0,
  'Chamae

In [13]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8414, Acc: 4.72% | Val Loss: 2.8527, Acc: 3.77% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.77%)


Epoch 2/50 | Train Loss: 2.8309, Acc: 6.79% | Val Loss: 2.8304, Acc: 5.54% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.54%)


Epoch 3/50 | Train Loss: 2.8005, Acc: 11.96% | Val Loss: 2.7918, Acc: 8.08% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.08%)


Epoch 4/50 | Train Loss: 2.7542, Acc: 19.74% | Val Loss: 2.7351, Acc: 17.50% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 17.50%)


Epoch 5/50 | Train Loss: 2.6810, Acc: 29.21% | Val Loss: 2.6490, Acc: 29.24% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 29.24%)


... (42 outputs left out)

Epoch 48/50 | Train Loss: 0.2686, Acc: 93.54% | Val Loss: 0.3824, Acc: 89.26% | LR: 4.87e-07[EarlyStop 1/10]


Epoch 49/50 | Train Loss: 0.2704, Acc: 93.72% | Val Loss: 0.3897, Acc: 89.26% | LR: 1.22e-07[EarlyStop 2/10]


Epoch 50/50 | Train Loss: 0.2701, Acc: 93.42% | Val Loss: 0.3740, Acc: 90.25% | LR: 0.00e+00[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 90.25%)
[taskB_label_smoothing_0.0 | resnet50 s42] val_acc=90.25% test_acc=93.42% F1=93.63%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 90.2547065337763,
 'best_epoch': 50,
 'final_train_acc': 93.41947115384616,
 'final_val_acc': 90.2547065337763,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9342105263157895,
 'test_f1_macro': 0.9362766048805454,
 'test_precision_macro': 0.9417018014727022,
 'test_recall_macro': 0.9360204317495868,
 'test_f1_per_class': {'Araucaria araucana': 0.9655172413793104,
  'Callitris columellaris': 0.927536231884058,
  'Calocedrus rupe

In [14]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | resnet50 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8402, Acc: 4.72% | Val Loss: 2.8491, Acc: 3.77% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 3.77%)


Epoch 2/50 | Train Loss: 2.8318, Acc: 6.64% | Val Loss: 2.8314, Acc: 5.65% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 5.65%)


Epoch 3/50 | Train Loss: 2.8078, Acc: 12.23% | Val Loss: 2.8014, Acc: 8.42% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 8.42%)


Epoch 4/50 | Train Loss: 2.7721, Acc: 20.67% | Val Loss: 2.7581, Acc: 18.72% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 18.72%)


Epoch 5/50 | Train Loss: 2.7159, Acc: 30.62% | Val Loss: 2.6937, Acc: 30.90% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to resnet50_seed42_best.pth (Val Acc: 30.90%)


... (42 outputs left out)

Epoch 48/50 | Train Loss: 1.2208, Acc: 93.84% | Val Loss: 1.2977, Acc: 89.26% | LR: 4.87e-07[EarlyStop 1/10]


Epoch 49/50 | Train Loss: 1.2230, Acc: 94.23% | Val Loss: 1.3004, Acc: 89.70% | LR: 1.22e-07[EarlyStop 2/10]


Epoch 50/50 | Train Loss: 1.2250, Acc: 93.69% | Val Loss: 1.2920, Acc: 90.37% | LR: 0.00e+00[EarlyStop 3/10]
[taskB_label_smoothing_0.2 | resnet50 s42] val_acc=90.48% test_acc=93.80% F1=94.00%


{'backbone': 'resnet50',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 90.47619047619048,
 'best_epoch': 47,
 'final_train_acc': 93.68990384615384,
 'final_val_acc': 90.36544850498339,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9379699248120301,
 'test_f1_macro': 0.9400233476993866,
 'test_precision_macro': 0.9454752355487649,
 'test_recall_macro': 0.9391633248362516,
 'test_f1_per_class': {'Araucaria araucana': 0.9830508474576272,
  'Callitris columellaris': 0.9565217391304348,
  'Calocedrus r

In [15]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    90.48%    93.98%    94.17%    94.65%    94.13%
main                              123    90.25%    91.35%    91.42%    92.59%    91.65%
main                              456    90.14%    91.73%    92.05%    93.10%    92.10%
taskB_90rotation                   42    92.14%    95.86%    95.80%    96.39%    95.72%
taskB_label_smoothing_0.0          42    90.25%    93.42%    93.63%    94.17%    93.60%
taskB_label_smoothing_0.2          42    90.48%    93.80%    94.00%    94.55%    93.92%
----------------------------------------------------------------------------------------
main test_acc: 92.36 +/- 1.16%
main test_f1:  92.55 +/- 1.18%
